In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

import pandas as pd
import seaborn as sns
import torch

device = torch.device("cuda")

os.environ["TRITON_PRINT_AUTOTUNING"] = "1"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["TRITON_INTERPRET"] = "1"

# Fused MM-Sample
* **Gumbel-Max Trick:** Sampling from the mulitnomial is equivalent to taking the argmax over logits plus standard Gumbel noise.
* We now attempt to sample from the logits without materializing them.
* We compute the logits incrementally, and as we do, we keep track of the gumbel max index.

In [ ]:
from fused_mm_sampling.testing import make_synthetic_inputs

inputs = make_synthetic_inputs(vocab_size=100, hidden_size=10)
weights = inputs.weights
hidden_states = inputs.hidden_states
vocab_size = inputs.vocab_size
n_hidden_states = inputs.logits.shape[0]

## Baseline: PyTorch Sampling

In [ ]:
from fused_mm_sampling.core import sample


def plot_samples(samples: torch.Tensor, n_hidden_states: int, num_samples: int):
    data = {
        "sample": samples.flatten().cpu(),
        "seq": [seq for seq in range(n_hidden_states) for _ in range(num_samples)],
    }
    df = pd.DataFrame(data)
    ax = sns.histplot(df, x="sample", hue="seq", bins=100)
    ax.grid(axis="y", alpha=0.5)


num_samples = 1000
temperature = 5
samples, probs = sample(
    weights,
    hidden_states,
    num_samples=num_samples,
    temperature=temperature,
    return_probs=True,
    seed=0,
    tl_matmul=False,
)  # [n_hidden_states, num_samples]
plot_samples(samples, n_hidden_states, num_samples)

## Fused PyTorch Incremental Sampling

In [ ]:
from fused_mm_sampling.core import sequential_sample_pt

samples2 = sequential_sample_pt(weights, hidden_states, num_samples, temperature)
plot_samples(samples2, n_hidden_states, num_samples)

# Triton

In [ ]:
from fused_mm_sampling import fused_mm_sample_triton

samples3 = fused_mm_sample_triton(
    weights,
    hidden_states,
    num_samples,
    temperature,  # temperature,
    seed=0,
)
plot_samples(samples3, n_hidden_states, num_samples)

# Helion

In [ ]:
from fused_mm_sampling.helion_impl import fused_mm_sample_helion

samples_helion = fused_mm_sample_helion(weights, hidden_states, num_samples, temperature)
plot_samples(samples_helion, n_hidden_states, num_samples)

# JL Dimensionality Reduction
Using the Johnson-Lindenstrauss Lemma, we can reduce the dimensionality of the innner product between the hidden states and the rows of the weights matrix.

In [ ]:
from fused_mm_sampling.core import JLSampler

sampler = JLSampler(weights, k=500).prepare()

torch.manual_seed(0)
plot_samples(
    samples=sampler.sample(
        hidden_states,
        temperature=temperature,
        num_samples=num_samples,
    ),
    n_hidden_states=n_hidden_states,
    num_samples=num_samples,
)

# Flashinfer

In [ ]:
from fused_mm_sampling.core import (
    flashinfer_sampling_from_logits,
    flashinfer_top_k_top_p_sampling_from_logits,
)

plot_samples(
    samples=flashinfer_top_k_top_p_sampling_from_logits(
        weights=weights,
        hidden_states=hidden_states,
        num_samples=num_samples,
        temperature=temperature,
        top_p=1.0,  # no restriction
        top_k=vocab_size,
    ),
    n_hidden_states=n_hidden_states,
    num_samples=num_samples,
)

In [ ]:
plot_samples(
    samples=flashinfer_sampling_from_logits(
        weights=weights,
        hidden_states=hidden_states,
        num_samples=num_samples,
        temperature=temperature,
    ),
    n_hidden_states=n_hidden_states,
    num_samples=num_samples,
)

# Memory Calculation

In [ ]:
def print_memory_usage():
    vocab_size = 250_000
    hidden_size = 8_000

    logits_numel = vocab_size * 256  # new_seqlen
    logits_bytes = logits_numel * 2  # bfloat16
    logits_gb = logits_bytes / 10**9
    print(f"logits_numel: {logits_numel:,}")
    print(f"logits_gb: {logits_gb:.2f} GB")

    weights_numel = vocab_size * hidden_size
    weights_bytes = weights_numel * 2  # bfloat16
    weights_gb = weights_bytes / 10**9
    print(f"weights_numel: {weights_numel:,}")
    print(f"weights_gb: {weights_gb:.2f} GB")


print_memory_usage()